# 09 — Mem0 Historical Memory

**Objective**: persist and retrieve `ProjectSnapshot` history (Section 11), then use that history to compute two things a single snapshot can never honestly answer on its own (Accuracy Check 6): week-over-week trend direction (Section 14) and multi-sprint blocker persistence (Section 12).

**Dependencies**: `src/services/memory_store.py`, `src/services/snapshot_builder.py`, `src/services/trend_engine.py`.

**Configuration**: `FileMemoryStore` (JSON-backed, no external services) is what this whole notebook runs against — it's what every test in this codebase actually exercises. `Mem0MemoryStore` (wrapping the real `mem0.Memory()` OSS client) is implemented for real but requires `OPENAI_API_KEY`, which isn't configured in this environment; the last section shows exactly what that dependency looks like and proves it via dependency injection instead.

In [1]:
import os
import sys
import shutil
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

from datetime import date

from src.connectors.jira_client import build_default_jira_client
from src.connectors.financial_client import CSVFinancialDataSource
from src.services import project_unifier, risk_engine, snapshot_builder, trend_engine, sprint_metrics
from src.services.memory_store import FileMemoryStore, MemoryScope, snapshot_to_fact_text

jira_client = build_default_jira_client()
fin_source = CSVFinancialDataSource()
mapping = project_unifier.load_project_mapping()
scope = MemoryScope(mapping.organization_id, mapping.portfolio_id)

NOTEBOOK_SNAPSHOT_DIR = PROJECT_ROOT / "data/snapshots"
shutil.rmtree(NOTEBOOK_SNAPSHOT_DIR, ignore_errors=True)  # clean slate for a reproducible run
store = FileMemoryStore(base_dir=NOTEBOOK_SNAPSHOT_DIR)

print(f"Scope: organization_id={scope.organization_id}, portfolio_id={scope.portfolio_id}")

Scope: organization_id=org-northwind, portfolio_id=portfolio-elt-2026


## Building a snapshot

One helper that mirrors what `persist_snapshot` (Phase 8's graph node) will do each run: unify the project, assess its risk, then assemble a `ProjectSnapshot`. `financial_record` is passed explicitly rather than read off `Project` — see `snapshot_builder.build_snapshot`'s docstring for why that matters for historical replay specifically.

In [2]:
def make_snapshot(entry, jira_as_of, finance_period, snap_date):
    issues, latest_sprint = None, None
    if entry.jira_key:
        issues = jira_client.get_project_issues(entry.jira_key, as_of=jira_as_of).records
        prev = jira_client.get_previous_sprints(entry.jira_key, 1, as_of=jira_as_of)
        latest_sprint = prev[0] if prev else None

    financial_record = None
    if entry.finance_project_id:
        raw = fin_source.get_project_finances(entry.finance_project_id, finance_period)
        financial_record = raw.with_calculated_fields() if raw else None

    project = project_unifier.build_unified_project(entry, jira_client, fin_source, jira_as_of)
    assessment = risk_engine.assess_project_risk(project, issues, latest_sprint, financial_record)
    return snapshot_builder.build_snapshot(project, assessment, issues, latest_sprint, financial_record, snap_date)

## Week-over-week trend, on real historical movement

PHX and TITAN both have real Sprint 2 -> Sprint 3 windows in the sample data (2026-01-19 to 2026-02-15) — not fabricated before/after numbers. PHX's completion genuinely jumped 25% -> 80%; TITAN's genuinely dropped 58.2% -> 40.6%. "Week 1"/"week 2" financial periods use the real Feb/Mar closes rather than being derived from `as_of` (a live pipeline would always want the true latest period; replaying two specific past months is a deliberate notebook-level choice, documented here rather than baked into `get_latest_reporting_period`).

In [3]:
phx_entry = mapping.entries["PROJECT-10001"]
titan_entry = mapping.entries["PROJECT-10004"]

week1_phx = make_snapshot(phx_entry, date(2026, 2, 5), "2026-02", date(2026, 2, 5))
week2_phx = make_snapshot(phx_entry, date(2026, 2, 20), "2026-03", date(2026, 2, 20))
week1_titan = make_snapshot(titan_entry, date(2026, 2, 5), "2026-02", date(2026, 2, 5))
week2_titan = make_snapshot(titan_entry, date(2026, 2, 20), "2026-03", date(2026, 2, 20))

for s in [week1_phx, week2_phx, week1_titan, week2_titan]:
    store.add_snapshot(scope, s)

print("PHX  ", week1_phx.sprint_completion_pct, "->", week2_phx.sprint_completion_pct, " RAG:", week1_phx.rag_status.value, "->", week2_phx.rag_status.value)
print("TITAN", week1_titan.sprint_completion_pct, "->", week2_titan.sprint_completion_pct, " RAG:", week1_titan.rag_status.value, "->", week2_titan.rag_status.value)

PHX   25.0 -> 80.0  RAG: RED -> RED
TITAN 58.18181818181818 -> 40.625  RAG: RED -> RED


In [4]:
phx_diff = trend_engine.diff_snapshots(week1_phx, week2_phx)
titan_diff = trend_engine.diff_snapshots(week1_titan, week2_titan)

print("PHX trend:  ", trend_engine.classify_trend(week1_phx, week2_phx).value)
print("  ", phx_diff)
print()
print("TITAN trend:", trend_engine.classify_trend(week1_titan, week2_titan).value)
print("  ", titan_diff)

PHX trend:   IMPROVING
   blocked_issue_delta=0 sprint_completion_delta=55.0 budget_consumption_delta=-1.3035521925431794 remaining_budget_delta=1702.3999999999942 risk_score_delta=0.0 open_issue_delta=0 rag_status_change=None

TITAN trend: DETERIORATING
   blocked_issue_delta=0 sprint_completion_delta=-17.55681818181818 budget_consumption_delta=-0.09552970374765835 remaining_budget_delta=3840.190000000017 risk_score_delta=0.0 open_issue_delta=0 rag_status_change=None


Both projects stay RED both weeks — the combined RAG doesn't move. But the trend engine correctly reports PHX as **IMPROVING** and TITAN as **DETERIORATING** underneath that unchanged label, exactly the nuance Section 14 exists to surface ("still RED, but here's whether it's getting better or worse").

## Accuracy Check 6, demonstrated: BASELINE with only one snapshot

In [5]:
only_one_snapshot = week2_phx
trend = trend_engine.classify_trend(None, only_one_snapshot)
print(f"Trend with no prior snapshot: {trend.value}  (never IMPROVING/DETERIORATING/STABLE from one data point)")

Trend with no prior snapshot: BASELINE  (never IMPROVING/DETERIORATING/STABLE from one data point)


## Section 12: "what has been stuck for more than one sprint?"

Requires >=2 stored snapshots per issue, computed from `major_blockers` history — never inferred from the current pull alone. Three consecutive weekly snapshots of PHX, all using the same real blocked-issue set (this dataset's blocker flags don't change week to week, which is itself realistic: a blocker nobody's touched stays a blocker).

In [6]:
s1 = make_snapshot(phx_entry, date(2026, 3, 1), "2026-01", date(2026, 3, 1))
s2 = make_snapshot(phx_entry, date(2026, 3, 8), "2026-02", date(2026, 3, 8))
s3 = make_snapshot(phx_entry, date(2026, 3, 15), "2026-03", date(2026, 3, 15))

print("major_blockers each week:")
for s in [s1, s2, s3]:
    print(f"  {s.snapshot_date}: {s.major_blockers}")

print()
print("With only s1 (1 snapshot):", trend_engine.find_persistent_blockers([s1]), "— cannot determine yet")
print("With s1, s2, s3 (3 snapshots):", trend_engine.find_persistent_blockers([s1, s2, s3]))
print()
print("PHX-17 sprint_count_blocked with 1 snapshot:", trend_engine.compute_sprint_count_blocked("PHX-17", [s1]))
print("PHX-17 sprint_count_blocked with 3 snapshots:", trend_engine.compute_sprint_count_blocked("PHX-17", [s1, s2, s3]))

major_blockers each week:
  2026-03-01: ['PHX-15', 'PHX-17', 'PHX-26', 'PHX-4']
  2026-03-08: ['PHX-15', 'PHX-17', 'PHX-26', 'PHX-4']
  2026-03-15: ['PHX-15', 'PHX-17', 'PHX-26', 'PHX-4']

With only s1 (1 snapshot): [] — cannot determine yet
With s1, s2, s3 (3 snapshots): ['PHX-15', 'PHX-17', 'PHX-26', 'PHX-4']

PHX-17 sprint_count_blocked with 1 snapshot: None
PHX-17 sprint_count_blocked with 3 snapshots: 3


## Closing the Phase 6 -> Phase 7 loop

`delivery_metrics.detect_blocked_multiple_sprints` (Phase 6) was written against `JiraIssue.sprint_count_blocked` before any history existed to populate it — it was a guaranteed no-op until this phase. `sprint_metrics.apply_sprint_count_blocked` (new this phase) is what finally fills that field in, using the snapshot history above.

In [7]:
from src.services import delivery_metrics

issues = jira_client.get_project_issues("PHX", as_of=date(2026, 3, 15)).records
issues_with_history = sprint_metrics.apply_sprint_count_blocked(issues, [s1, s2, s3])

for i in issues_with_history:
    if i.blocked:
        print(f"  {i.issue_key}: sprint_count_blocked={i.sprint_count_blocked}")

finding = delivery_metrics.detect_blocked_multiple_sprints("PHX", issues_with_history, delivery_metrics.load_delivery_risk_rules())
print()
print(f"detect_blocked_multiple_sprints now fires: {finding.severity.value if finding else None}")
print(f"  {finding.description}")

  PHX-4: sprint_count_blocked=3
  PHX-15: sprint_count_blocked=3
  PHX-17: sprint_count_blocked=3
  PHX-26: sprint_count_blocked=3

detect_blocked_multiple_sprints now fires: HIGH
  4 issue(s) blocked across 2+ sprints


## What gets stored: fact text vs. structured metadata

The fact text (Section 11's example format) is for human/LLM readability only. Every actual calculation above worked off the structured `ProjectSnapshot` fields, never by re-parsing this string.

In [8]:
print(snapshot_to_fact_text(week2_phx))

Project PROJECT-10001, as of 2026-02-20, sprint PHX-SPR-3, completion 80.0%, blocked issues: 4, budget consumed: 110.9%, combined risk: RED


## Scope isolation

In [9]:
other_scope = MemoryScope(organization_id="org-different-tenant", portfolio_id=mapping.portfolio_id)
print(f"PHX history under the real scope:  {len(store.get_snapshots(scope, 'PROJECT-10001'))} snapshots")
print(f"PHX history under a different org: {len(store.get_snapshots(other_scope, 'PROJECT-10001'))} snapshots (isolated)")

PHX history under the real scope:  2 snapshots
PHX history under a different org: 0 snapshots (isolated)


## Mem0MemoryStore: real implementation, verified requirements, dependency-injected proof

The exact method signatures below were checked with `inspect.signature()` against the installed `mem0ai` package, not taken from documentation alone.

In [10]:
import inspect
from mem0 import Memory

print("Memory.add    ", inspect.signature(Memory.add))
print("Memory.get_all", inspect.signature(Memory.get_all))

Memory.add     (self, messages, *, user_id: str | None = None, agent_id: str | None = None, run_id: str | None = None, metadata: Dict[str, Any] | None = None, timestamp: Any | None = None, expiration_date: Any | None = None, infer: bool = True, memory_type: str | None = None, prompt: str | None = None)
Memory.get_all (self, *, filters: Dict[str, Any] | None = None, top_k: int = 20, show_expired: bool = False, **kwargs)


In [11]:
# Confirm the documented requirement actually holds: Memory() needs OPENAI_API_KEY
# even in "local" OSS mode, since its default embedder/LLM is OpenAI.
from src.services.memory_store import Mem0MemoryStore

saved_key = os.environ.pop("OPENAI_API_KEY", None)
try:
    Mem0MemoryStore()
    print("Unexpected: constructed without OPENAI_API_KEY")
except Exception as e:
    print(f"Confirmed: {type(e).__name__}: {str(e)[:120]}")
finally:
    if saved_key:
        os.environ["OPENAI_API_KEY"] = saved_key

Confirmed: OpenAIError: Missing credentials. Please pass an `api_key`, `workload_identity`, `admin_api_key`, or set the `OPENAI_API_KEY` or `OPE


In [12]:
# Dependency injection: prove the org/portfolio/project -> user_id/agent_id/run_id
# mapping and infer=False are wired correctly, without needing OPENAI_API_KEY.
class FakeMem0Client:
    def __init__(self):
        self.calls = []
    def add(self, messages, **kwargs):
        self.calls.append((messages, kwargs))
        return {"results": [{"id": "fake-memory-id"}]}
    def get_all(self, **kwargs):
        return {"results": []}

fake_client = FakeMem0Client()
mem0_store = Mem0MemoryStore(memory=fake_client)
memory_id = mem0_store.add_snapshot(scope, week2_phx)

fact_text, kwargs = fake_client.calls[0]
print(f"memory_id returned: {memory_id}")
print(f"fact_text passed:   {fact_text}")
print(f"user_id={kwargs['user_id']}  agent_id={kwargs['agent_id']}  run_id={kwargs['run_id']}")
print(f"infer={kwargs['infer']}  (False — Mem0's own LLM must never rewrite these numbers)")
print(f"metadata keys: {sorted(kwargs['metadata'].keys())[:5]}...")

memory_id returned: fake-memory-id
fact_text passed:   Project PROJECT-10001, as of 2026-02-20, sprint PHX-SPR-3, completion 80.0%, blocked issues: 4, budget consumed: 110.9%, combined risk: RED
user_id=org-northwind  agent_id=portfolio-elt-2026  run_id=PROJECT-10001
infer=False  (False — Mem0's own LLM must never rewrite these numbers)
metadata keys: ['actual_spend', 'approved_budget', 'blocked_issues', 'budget_consumption_pct', 'delivery_progress_pct']...


## Validation checks

- [x] `FileMemoryStore` round-trips a snapshot exactly, chronologically ordered, with scope and project isolation
- [x] Trend classification uses real historical sprint/financial movement (PHX genuinely improved, TITAN genuinely deteriorated) — not fabricated before/after numbers
- [x] A single snapshot always yields `BASELINE`/`None`/`[]` for trend/count/persistence — never inferred from one data point (Accuracy Check 6)
- [x] Phase 6's `detect_blocked_multiple_sprints`, a guaranteed no-op until now, correctly fires once `apply_sprint_count_blocked` populates real history
- [x] `Mem0MemoryStore`'s documented `OPENAI_API_KEY` requirement is proven live, not just asserted in a docstring
- [x] The org/portfolio/project -> user_id/agent_id/run_id mapping and `infer=False` are proven correct via dependency injection, without needing real Mem0 credentials

## Testing

`tests/test_memory_store.py` (14 tests), `tests/test_trend_engine.py` (18 tests), `tests/test_snapshot_builder.py` (5 tests), plus new coverage in `tests/test_risk_engine.py` (`compute_risk_score`) and `tests/test_sprint_metrics.py` (`apply_sprint_count_blocked`).

## Next step

Phase 8: LangGraph orchestration — wiring `src/graph/nodes.py`'s stubs (`retrieve_historical_memory`, `persist_snapshot`, and everything upstream) into an actual `StateGraph`, using `memory_agent.py`'s two functions as the graph's memory-facing calls.